# 🛒 Maven Fuzzy Factory – Data Cleaning & EDA
> **Author:** Binh Pham | **Dataset:** Maven Fuzzy Factory (E-Commerce)

This notebook covers data loading, cleaning, and an initial exploratory overview of all six tables.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5),
                     'axes.titlesize': 14, 'axes.titleweight': 'bold'})

BASE = '..'
RAW  = f'{BASE}/data/raw'
PROC = f'{BASE}/data/processed'


## 1. Load Raw Data

In [ ]:

orders    = pd.read_csv(f'{RAW}/orders.csv',            parse_dates=['created_at'])
items     = pd.read_csv(f'{RAW}/order_items.csv',       parse_dates=['created_at'])
refunds   = pd.read_csv(f'{RAW}/order_item_refunds.csv',parse_dates=['created_at'])
products  = pd.read_csv(f'{RAW}/products.csv',          parse_dates=['created_at'])
sessions  = pd.read_csv(f'{RAW}/website_sessions.csv',  parse_dates=['created_at'])

tables = {'orders': orders, 'order_items': items, 'refunds': refunds,
          'products': products, 'sessions': sessions}

for name, df in tables.items():
    print(f"{name:20s}  {df.shape[0]:>10,} rows  {df.shape[1]:>3} cols  "
          f"  nulls: {df.isnull().sum().sum()}")


## 2. Cleaning Steps

In [ ]:

# Orders – derived columns
orders['year_month']   = orders['created_at'].dt.to_period('M').astype(str)
orders['quarter']      = orders['created_at'].dt.to_period('Q').astype(str)
orders['gross_profit'] = (orders['price_usd'] - orders['cogs_usd']).round(2)
orders['margin_pct']   = (orders['gross_profit'] / orders['price_usd'] * 100).round(2)

# Sessions – fill UTM nulls
sessions['utm_source']   = sessions['utm_source'].fillna('direct')
sessions['utm_campaign'] = sessions['utm_campaign'].fillna('none')
sessions['utm_content']  = sessions['utm_content'].fillna('none')
sessions['device_type']  = sessions['device_type'].str.lower().str.strip()
sessions['year_month']   = sessions['created_at'].dt.to_period('M').astype(str)

# Items
items['year_month']   = items['created_at'].dt.to_period('M').astype(str)
items['gross_profit'] = (items['price_usd'] - items['cogs_usd']).round(2)

print("✔  Cleaning complete")
print(f"Date range (orders): {orders['created_at'].min().date()} → {orders['created_at'].max().date()}")
print(f"Unique customers: {orders['user_id'].nunique():,}")
print(f"Total revenue: ${orders['price_usd'].sum():,.2f}")


## 3. Schema at a Glance

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, (name, df) in zip(axes, tables.items()):
    missing_pct = df.isnull().mean() * 100
    colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in missing_pct]
    ax.barh(missing_pct.index, missing_pct.values, color=colors)
    ax.set_title(f'{name}  ({df.shape[0]:,} rows)')
    ax.set_xlabel('Missing %')
    ax.axvline(0, color='black', linewidth=0.5)

fig.suptitle('Missing Value Overview by Table', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_missing_values.png', bbox_inches='tight')
plt.show()


## 4. Key Dataset Stats

In [ ]:

summary = {
    'Total Sessions':     f"{sessions.shape[0]:,}",
    'Total Orders':       f"{orders.shape[0]:,}",
    'Total Revenue':      f"${orders['price_usd'].sum():,.0f}",
    'Gross Profit':       f"${orders['gross_profit'].sum():,.0f}",
    'Avg Margin':         f"{orders['margin_pct'].mean():.1f}%",
    'Unique Customers':   f"{orders['user_id'].nunique():,}",
    'Total Refunds':      f"{refunds.shape[0]:,}",
    'Overall Refund Rate':f"{refunds.shape[0]/items.shape[0]*100:.2f}%",
    'Products':           str(products.shape[0]),
    'Date Range':         f"{orders['created_at'].min().date()} → {orders['created_at'].max().date()}"
}
for k, v in summary.items():
    print(f"  {k:<25} {v}")
